In [3]:
import torch
from kerops.ops.conv.conv_wgrad import Conv3dWgrad, conv3d_wgrad_config, autotune_conv_wgrad

Conv3dWgrad.register_kernel_config(conv3d_wgrad_config)

cin = 128
cout = 128
S = D = 64
x = torch.randn(1, cin, S, S, D, device='cuda', dtype=torch.float16).to(memory_format=torch.channels_last_3d)
grad = torch.randn(1, cout, S, S, D, device='cuda', dtype=torch.float16).to(memory_format=torch.channels_last_3d)

In [5]:
%%timeit -r 10 -n 10
Conv3dWgrad(grad, x)
torch.cuda.synchronize()

5.03 ms ± 54.2 μs per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [7]:
weight = torch.randn(cout, cin, 3, 3, 3, device='cuda', dtype=torch.float16)

In [9]:
%%timeit -r 10 -n 10
torch.ops.aten.convolution_backward(
    grad,
    x,
    weight,
    [0],  # bias_sizes
    [1, 1, 1],  # stride
    [1, 1, 1],  # padding
    [1, 1, 1],  # dilation
    False,  # transposed
    [0, 0, 0],  # output padding
    1,  # groups!
    [False, True, False],  # output_mask - grad_inpt, grad_weight, grad_bias
)
torch.cuda.synchronize()

3.71 ms ± 67.9 μs per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [10]:
import triton
from triton import language as tl, next_power_of_2


@triton.jit
def make_offset(h_str, w_str, d_str, H_BLOCK: tl.constexpr, W_BLOCK: tl.constexpr, D_BLOCK: tl.constexpr):
    d_off = tl.arange(0, D_BLOCK)
    w_off = tl.arange(0, W_BLOCK)
    h_off = tl.arange(0, H_BLOCK)

    offset = h_off[:, None, None] * h_str + w_off[None, :, None] * w_str + d_off[None, None, :] * d_str
    offset = offset.reshape((H_BLOCK * W_BLOCK * D_BLOCK))

    return offset


@triton.jit
def make_mask(curr_h, curr_w, curr_d, H, W, D, H_BLOCK: tl.constexpr, W_BLOCK: tl.constexpr, D_BLOCK: tl.constexpr):
    mask_d = ((tl.arange(0, D_BLOCK) + curr_d) >= 0) & ((tl.arange(0, D_BLOCK) + curr_d) < D)
    mask_w = ((tl.arange(0, W_BLOCK) + curr_w) >= 0) & ((tl.arange(0, W_BLOCK) + curr_w) < W)
    mask_h = ((tl.arange(0, H_BLOCK) + curr_h) >= 0) & ((tl.arange(0, H_BLOCK) + curr_h) < H)

    mask = mask_h[:, None, None] & mask_w[None, :, None] & mask_d[None, None, :]
    mask = mask.reshape((H_BLOCK * W_BLOCK * D_BLOCK))

    return mask


@triton.jit
def _Conv_wgrad_cl3d_impl(
    grad_ptr,
    input_ptr,
    weight_grad_ptr,
    H,
    W,
    D,
    ACCTYPE: tl.constexpr,
    H_BLOCK: tl.constexpr, W_BLOCK: tl.constexpr, D_BLOCK: tl.constexpr,
    IN_CHANNELS: tl.constexpr, OUT_CHANNELS: tl.constexpr,
    CIN_BLOCK: tl.constexpr, COUT_BLOCK: tl.constexpr,
    SPLIT_K: tl.constexpr,
):
    khwd_pid = tl.program_id(0)
    cin_pid = tl.program_id(1)
    cout_pid = tl.program_id(2)

    k_pid = khwd_pid // 27
    hwd_pid = khwd_pid % 27

    block_d = hwd_pid % 3
    hwd_pid = hwd_pid // 3
    block_w = hwd_pid % 3
    block_h = hwd_pid // 3

    cin_offset = tl.arange(0, CIN_BLOCK)
    cout_offset = tl.arange(0, COUT_BLOCK)
    geom_offset = make_offset(W * D, D, 1, H_BLOCK, W_BLOCK, D_BLOCK)

    grad_offset = geom_offset[:, None] * OUT_CHANNELS + cout_offset[None, :]
    input_offset = geom_offset[None, :] * IN_CHANNELS + cin_offset[:, None]
    weight_grad_offset = cin_offset[:, None] * OUT_CHANNELS + cout_offset[None, :]

    grad_ptr += cout_pid * COUT_BLOCK
    grad_ptr  += k_pid * H_BLOCK * OUT_CHANNELS * D * W
    input_ptr += cin_pid * CIN_BLOCK
    input_ptr += (block_d - 1) * IN_CHANNELS
    input_ptr += k_pid * H_BLOCK * IN_CHANNELS * D * W
    input_ptr += (block_w - 1) * IN_CHANNELS * D
    input_ptr += (block_h - 1) * IN_CHANNELS * D * W
    weight_grad_ptr += cin_pid * CIN_BLOCK * OUT_CHANNELS + cout_pid * COUT_BLOCK
    weight_grad_ptr += block_d * IN_CHANNELS * OUT_CHANNELS
    weight_grad_ptr += block_w * IN_CHANNELS * OUT_CHANNELS * 3
    weight_grad_ptr += block_h * IN_CHANNELS * OUT_CHANNELS * 9

    weight_grad = tl.zeros((CIN_BLOCK, COUT_BLOCK), dtype=ACCTYPE)

    for grad_h in range(0, tl.cdiv(H, H_BLOCK * SPLIT_K)):
        for grad_w in range(0, tl.cdiv(W, W_BLOCK)):
            for grad_d in range(0, tl.cdiv(D, D_BLOCK)):
                grad_mask = make_mask(grad_h * H_BLOCK * SPLIT_K + k_pid * H_BLOCK, grad_w * W_BLOCK, grad_d * D_BLOCK, H, W, D, H_BLOCK, W_BLOCK, D_BLOCK)
                grad_iter_ptr = (
                    grad_ptr
                    + grad_h * H_BLOCK * W * D * OUT_CHANNELS * SPLIT_K
                    + grad_w * W_BLOCK * D * OUT_CHANNELS
                    + grad_d * D_BLOCK * OUT_CHANNELS
                )
                grad = tl.load(grad_iter_ptr + grad_offset, mask=grad_mask[:, None], other=0)

                x_mask = make_mask(grad_h * H_BLOCK * SPLIT_K + k_pid * H_BLOCK + block_h - 1, grad_w * W_BLOCK + block_w - 1, grad_d * D_BLOCK + block_d - 1, H, W, D, H_BLOCK, W_BLOCK, D_BLOCK)
                x_iter_ptr = (
                    input_ptr
                    + grad_h * H_BLOCK * W * D * IN_CHANNELS * SPLIT_K
                    + grad_w * W_BLOCK * D * IN_CHANNELS
                    + grad_d * D_BLOCK * IN_CHANNELS
                )
                x = tl.load(x_iter_ptr + input_offset, mask=x_mask[None, :], other=0)

                weight_grad += tl.dot(x, grad)

    tl.atomic_add(weight_grad_ptr + weight_grad_offset, weight_grad, sem='relaxed')

In [63]:
@triton.jit
def _Conv_wgrad_cl3d_impl_prefetch(
    grad_ptr,
    input_ptr,
    weight_grad_ptr,
    H,
    W,
    D,
    ACCTYPE: tl.constexpr,
    H_BLOCK: tl.constexpr, W_BLOCK: tl.constexpr, D_BLOCK: tl.constexpr,
    IN_CHANNELS: tl.constexpr, OUT_CHANNELS: tl.constexpr,
    CIN_BLOCK: tl.constexpr, COUT_BLOCK: tl.constexpr,
):
    hwd_pid = tl.program_id(0)
    cin_pid = tl.program_id(1)
    cout_pid = tl.program_id(2)

    #hwd_pid = tl.program_id(0)
    #cin_cout_pid = tl.program_id(1)

    block_d = hwd_pid % 3
    hwd_pid = hwd_pid // 3
    block_w = hwd_pid % 3
    block_h = hwd_pid // 3

    #cin_pid = cin_cout_pid % tl.cdiv(IN_CHANNELS, CIN_BLOCK)
    #cout_pid = cin_cout_pid // tl.cdiv(IN_CHANNELS, CIN_BLOCK)

    cin_offset = tl.arange(0, CIN_BLOCK)
    cout_offset = tl.arange(0, COUT_BLOCK)
    geom_offset = make_offset(W * D, D, 1, H_BLOCK, W_BLOCK, D_BLOCK)

    grad_offset = geom_offset[:, None] * OUT_CHANNELS + cout_offset[None, :]
    input_offset = geom_offset[None, :] * IN_CHANNELS + cin_offset[:, None]
    weight_grad_offset = cin_offset[:, None] * OUT_CHANNELS + cout_offset[None, :]

    grad_ptr += cout_pid * COUT_BLOCK
    input_ptr += cin_pid * CIN_BLOCK
    input_ptr += (block_d - 1) * IN_CHANNELS
    input_ptr += (block_w - 1) * IN_CHANNELS * D
    input_ptr += (block_h - 1) * IN_CHANNELS * D * W
    weight_grad_ptr += cin_pid * CIN_BLOCK * OUT_CHANNELS + cout_pid * COUT_BLOCK
    weight_grad_ptr += block_d * IN_CHANNELS * OUT_CHANNELS
    weight_grad_ptr += block_w * IN_CHANNELS * OUT_CHANNELS * 3
    weight_grad_ptr += block_h * IN_CHANNELS * OUT_CHANNELS * 9

    weight_grad = tl.zeros((CIN_BLOCK, COUT_BLOCK), dtype=tl.float32)
    
    grad_mask = make_mask(0, 0, 0, H, W, D, H_BLOCK, W_BLOCK, D_BLOCK)
    grad_next = tl.load(grad_ptr + grad_offset, mask=grad_mask[:, None], other=0)

    x_mask = make_mask(block_h - 1, block_w - 1, block_d - 1, H, W, D, H_BLOCK, W_BLOCK, D_BLOCK)
    x_next = tl.load(input_ptr + input_offset, mask=x_mask[None, :], other=0)

    h_bound = tl.cdiv(H, H_BLOCK)
    w_bound = tl.cdiv(W, W_BLOCK)
    d_bound = tl.cdiv(D, D_BLOCK)

    for grad_h in range(0, h_bound):
        for grad_w in range(0, w_bound):
            for grad_d in range(0, d_bound):
                grad = grad_next
                x = x_next

                if ((grad_h + 1) != h_bound) and ((grad_w + 1) != w_bound) and ((grad_d + 1) != d_bound):
                    grad_mask = make_mask(grad_h * H_BLOCK, grad_w * W_BLOCK, grad_d * D_BLOCK, H, W, D, H_BLOCK, W_BLOCK, D_BLOCK)
                    grad_iter_ptr = (
                        grad_ptr
                        + grad_h * H_BLOCK * W * D * OUT_CHANNELS
                        + grad_w * W_BLOCK * D * OUT_CHANNELS
                        + grad_d * D_BLOCK * OUT_CHANNELS
                    )
                    grad_next = tl.load(grad_iter_ptr + grad_offset, mask=grad_mask[:, None], other=0)
    
                    x_mask = make_mask(grad_h * H_BLOCK + block_h - 1, grad_w * W_BLOCK + block_w - 1, grad_d * D_BLOCK + block_d - 1, H, W, D, H_BLOCK, W_BLOCK, D_BLOCK)
                    x_iter_ptr = (
                        input_ptr
                        + grad_h * H_BLOCK * W * D * IN_CHANNELS
                        + grad_w * W_BLOCK * D * IN_CHANNELS
                        + grad_d * D_BLOCK * IN_CHANNELS
                    )
                    x_next = tl.load(x_iter_ptr + input_offset, mask=x_mask[None, :], other=0)

                weight_grad += tl.dot(x, grad)

    tl.store(weight_grad_ptr + weight_grad_offset, weight_grad)

In [11]:
from kerops.settings import autotune, ConfArg, TableKernelConfig, ConfiguredFunction
from kerops.utils import cdiv


def Conv3dWgrad_plain(
    grad,
    x,
    *,
    num_warps: ConfArg,
    H_BLOCK: ConfArg,
    W_BLOCK: ConfArg,
    D_BLOCK: ConfArg,
    CIN_BLOCK: ConfArg, 
    COUT_BLOCK: ConfArg,
    SPLIT_K: ConfArg,
):
    assert x.device == grad.device
    assert x.is_cuda

    assert x.ndim == grad.ndim == 5
    xbsize, in_channels, xH, xW, xD = x.shape
    gbsize, out_channels, gH, gW, gD = grad.shape
    assert in_channels == next_power_of_2(in_channels)
    assert out_channels == next_power_of_2(out_channels)
    assert [xbsize, xH, xW, xD] == [gbsize, gH, gW, gD]

    assert xbsize == gbsize == 1

    assert x.is_contiguous(memory_format=torch.channels_last_3d)
    assert grad.is_contiguous(memory_format=torch.channels_last_3d)

    assert x.dtype == grad.dtype == torch.float16

    assert D_BLOCK == next_power_of_2(D_BLOCK)
    assert W_BLOCK == next_power_of_2(W_BLOCK)
    assert D_BLOCK == next_power_of_2(D_BLOCK)

    ACCTYPE = tl.float32
    weight_grad = torch.zeros([3, 3, 3, in_channels, out_channels], device=x.device, dtype=torch.float32)

    grid = (
        27 * SPLIT_K,
        cdiv(in_channels, CIN_BLOCK),
        cdiv(out_channels, COUT_BLOCK)
    )

    _Conv_wgrad_cl3d_impl[grid](
        grad,
        x,
        weight_grad,
        xH,
        xW,
        xD,
        ACCTYPE=ACCTYPE,
        H_BLOCK=H_BLOCK, W_BLOCK=W_BLOCK, D_BLOCK=D_BLOCK,
        IN_CHANNELS=in_channels, OUT_CHANNELS=out_channels,
        CIN_BLOCK=CIN_BLOCK, COUT_BLOCK=COUT_BLOCK,
        SPLIT_K=SPLIT_K,
        num_warps=num_warps,
    )
    
    weight_grad = weight_grad.to(torch.float16)

    return weight_grad

In [12]:
print(grad.shape)
print(x.shape)

torch.Size([1, 128, 64, 64, 64])
torch.Size([1, 128, 64, 64, 64])


In [16]:
%%timeit -r 10 -n 10
Conv3dWgrad_plain(grad, x, num_warps=4, H_BLOCK=4, W_BLOCK=4, D_BLOCK=4, CIN_BLOCK=64, COUT_BLOCK=64, SPLIT_K=4)
torch.cuda.synchronize()

3.52 ms ± 86.1 μs per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [76]:
out_ = Conv3dWgrad_plain(grad, x, num_warps=4, H_BLOCK=4, W_BLOCK=4, D_BLOCK=4, CIN_BLOCK=64, COUT_BLOCK=64, SPLIT_K=4)
out = Conv3dWgrad(grad, x)

In [104]:
torch.abs(out_ - out).max()

tensor(8., device='cuda:0', dtype=torch.float16)

In [105]:
torch.abs(out_).max()

tensor(6480., device='cuda:0', dtype=torch.float16)